In [ ]:
# 시계열 데이터 Time Series Data 분석
# 이동평균 계산
# 변화율 계산 
# 센서 데이터 이상치 탐지
# OEE 계산하는 방식 - 공장,생산 설비가 얼마나 효율적으로 운영되는지 측정하는 KPI

In [4]:
import pandas as pd
import numpy as np

In [5]:
production_df = pd.read_csv('./data/05_production.csv', encoding='utf-8-sig')
quality_df = pd.read_csv('./data/07_quality_inspection.csv', encoding='utf-8-sig', na_values=['\\N'])    
sensor_df = pd.read_csv('./data/08_sensor_data.csv', encoding='utf-8-sig')
operation_df = pd.read_csv('./data/06_equipment_operation.csv', encoding='utf-8-sig')
equipment_df = pd.read_csv('./data/01_equipment.csv', encoding='utf-8-sig')

# 날짜/시간 변환
production_df['production_date'] = pd.to_datetime(production_df['production_date'])
production_df['start_time'] = pd.to_datetime(production_df['start_time'])
production_df['end_time'] = pd.to_datetime(production_df['end_time'])
sensor_df['measurement_time'] = pd.to_datetime(sensor_df['measurement_time'])
operation_df['start_time'] = pd.to_datetime(operation_df['start_time'])
operation_df['end_time'] = pd.to_datetime(operation_df['end_time'])

In [6]:
# 일별 생산량 추이 => 일별로 집계

In [9]:
daily_production = production_df.groupby('production_date').agg( {'production_id':'count',
                                              'actual_quantity' : 'sum',
                                              'defect_quantity' : 'sum'}  ) 

In [11]:
daily_production.columns = ['생산건수' , '총생산량', '총불량수']

In [15]:
daily_production['불량률'] = (daily_production['총불량수'] / daily_production['총생산량'] * 100).round(2)

In [21]:
production_df

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,77.63,WO202403317101,LOT2024033100210,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,76.15,WO202403318434,LOT2024033100211,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,69.95,WO202403317294,LOT2024033100212,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,90.10,WO202403317268,LOT2024033100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48


In [16]:
daily_production.head(3)

,생산건수,총생산량,총불량수,불량률
production_date,,,,
2024-01-01,20,2019,114,5.65
2024-01-02,22,2380,128,5.38
2024-01-03,16,1848,95,5.14


In [22]:
sensor_df

,sensor_id,equipment_id,measurement_time,temperature,pressure,vibration,current,voltage,rpm,created_at
0,1,INJ-001,2024-01-01 00:00:00,183.93,148.65,2.6838,48.05,218.83,1795.32,2026-01-30 00:45:52
1,2,INJ-002,2024-01-01 00:00:00,173.06,144.23,2.2576,39.17,211.38,1738.75,2026-01-30 00:45:52
2,3,PRESS-001,2024-01-01 00:00:00,82.29,202.09,3.4924,120.14,372.88,489.11,2026-01-30 00:45:52
3,4,PRESS-002,2024-01-01 00:00:00,87.62,194.68,3.7005,121.70,379.93,498.85,2026-01-30 00:45:52
4,5,ASM-001,2024-01-01 00:00:00,20.78,-0.94,1.1674,16.48,220.86,97.69,2026-01-30 00:45:52
...,...,...,...,...,...,...,...,...,...,...
10915,10916,INJ-001,2024-03-31 23:00:00,192.48,148.93,2.4576,48.58,226.34,1775.78,2026-01-30 00:45:52
10916,10917,INJ-002,2024-03-31 23:00:00,175.82,141.36,2.4609,44.68,226.68,1762.20,2026-01-30 00:45:52
10917,10918,PRESS-001,2024-03-31 23:00:00,84.57,191.96,3.5106,116.58,383.60,492.28,2026-01-30 00:45:52
10918,10919,PRESS-002,2024-03-31 23:00:00,87.53,202.47,3.8016,118.02,390.56,506.57,2026-01-30 00:45:52


In [19]:
sensor_df['measurement_time'].unique()

<DatetimeArray>
['2024-01-01 00:00:00', '2024-01-01 01:00:00', '2024-01-01 02:00:00',
 '2024-01-01 03:00:00', '2024-01-01 04:00:00', '2024-01-01 05:00:00',
 '2024-01-01 06:00:00', '2024-01-01 07:00:00', '2024-01-01 08:00:00',
 '2024-01-01 09:00:00',
 ...
 '2024-03-31 14:00:00', '2024-03-31 15:00:00', '2024-03-31 16:00:00',
 '2024-03-31 17:00:00', '2024-03-31 18:00:00', '2024-03-31 19:00:00',
 '2024-03-31 20:00:00', '2024-03-31 21:00:00', '2024-03-31 22:00:00',
 '2024-03-31 23:00:00']
Length: 2184, dtype: datetime64[ns]

In [20]:
sensor_df

,sensor_id,equipment_id,measurement_time,temperature,pressure,vibration,current,voltage,rpm,created_at
0,1,INJ-001,2024-01-01 00:00:00,183.93,148.65,2.6838,48.05,218.83,1795.32,2026-01-30 00:45:52
1,2,INJ-002,2024-01-01 00:00:00,173.06,144.23,2.2576,39.17,211.38,1738.75,2026-01-30 00:45:52
2,3,PRESS-001,2024-01-01 00:00:00,82.29,202.09,3.4924,120.14,372.88,489.11,2026-01-30 00:45:52
3,4,PRESS-002,2024-01-01 00:00:00,87.62,194.68,3.7005,121.70,379.93,498.85,2026-01-30 00:45:52
4,5,ASM-001,2024-01-01 00:00:00,20.78,-0.94,1.1674,16.48,220.86,97.69,2026-01-30 00:45:52
...,...,...,...,...,...,...,...,...,...,...
10915,10916,INJ-001,2024-03-31 23:00:00,192.48,148.93,2.4576,48.58,226.34,1775.78,2026-01-30 00:45:52
10916,10917,INJ-002,2024-03-31 23:00:00,175.82,141.36,2.4609,44.68,226.68,1762.20,2026-01-30 00:45:52
10917,10918,PRESS-001,2024-03-31 23:00:00,84.57,191.96,3.5106,116.58,383.60,492.28,2026-01-30 00:45:52
10918,10919,PRESS-002,2024-03-31 23:00:00,87.53,202.47,3.8016,118.02,390.56,506.57,2026-01-30 00:45:52


In [27]:
sensor_1 = sensor_df.loc[ sensor_df['equipment_id'] == 'INJ-001' , ].sort_values('measurement_time')

In [31]:
sensor_1.reset_index(drop= True, inplace=True)

In [37]:
sensor_1.columns

Index(['sensor_id', 'equipment_id', 'measurement_time', 'temperature',
       'pressure', 'vibration', 'current', 'voltage', 'rpm', 'created_at'],
      dtype='object')

In [43]:
# resample 함수를 이용해서, datetime 을 조절할때에는
# datetime 컬럼이 무조건 인덱스로 되어있어야 한다.
sensor_1.set_index('measurement_time')[ ['temperature',
       'pressure', 'vibration', 'current', 'voltage', 'rpm'] ].resample('ME').mean()

,temperature,pressure,vibration,current,voltage,rpm
measurement_time,,,,,,
2024-01-31,182.559113,150.072540,2.488544,45.015255,219.894798,1799.577594
2024-02-29,187.498865,150.023693,2.498557,44.946092,219.904986,1800.951264
2024-03-31,192.471989,149.969140,2.499791,45.037204,219.748602,1800.565806


In [47]:
sensor_all = sensor_df.groupby('measurement_time')[ ['temperature','pressure','vibration','current','voltage','rpm']].mean()         

In [51]:
sensor_all.resample('D').mean()

,temperature,pressure,vibration,current,voltage,rpm
measurement_time,,,,,,
2024-01-01,110.660667,139.611750,2.814983,68.186667,284.601583,932.055000
2024-01-02,111.031417,139.831500,2.686983,68.438750,284.228583,936.736667
2024-01-03,110.668000,140.153833,2.695356,68.191667,284.183250,932.576417
2024-01-04,110.718333,139.901167,2.712816,68.124250,284.065250,934.209417
2024-01-05,110.373833,139.653000,2.751715,68.151167,283.345417,934.190083
...,...,...,...,...,...,...
2024-03-27,113.264417,139.593083,2.667431,68.441833,285.089500,933.713583
2024-03-28,113.161583,139.867250,2.704023,68.069917,284.704333,935.672583
2024-03-29,113.754250,139.826000,2.767098,68.518333,284.464333,932.220750


In [52]:
production_df.head(2)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48


In [55]:
production_df['생산효율'] = (production_df['actual_quantity'] / production_df['target_quantity'] * 100).round(2)

In [56]:
production_df

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,생산효율
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,83.51
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,93.98
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,90.60
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,92.00
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,104.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,77.63,WO202403317101,LOT2024033100210,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,96.00
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,76.15,WO202403318434,LOT2024033100211,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,95.59
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,69.95,WO202403317294,LOT2024033100212,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,95.24
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,90.10,WO202403317268,LOT2024033100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,84.62


In [57]:
def get_grade(x):
    if x > 90 :
        return '우수'
    elif x > 80 :
        return '양호'
    else :
        return '부족'

In [60]:
production_df['효율 등급'] = production_df['생산효율'].apply(get_grade)

In [61]:
production_df.head(2)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,생산효율,효율 등급
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,83.51,양호
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,93.98,우수


In [62]:
# 이동평균 : 센서 데이터에는 노이즈가 많아서 추세 파악이 힘들다.
# 노이즈 제거나 이상패턴 감지가 쉽다.

In [63]:
# 진동센서 7일 이동평균이 상승 -> 베어링 교체
# 온도 30일 이동평균이 상승 -> 냉각 시스템 점검 
# 측정값 이동평균이 상한선 접근 -> 사전 경고를 해줄수 있다. 
# 생산량 이동평균으로 수요 예측
# 계절성 패턴 파악 가능

In [78]:
sensor_1_day = sensor_1.set_index('measurement_time').resample('D')['temperature'].mean()

In [81]:
sensor_1_day.rolling(window=5).mean().head(10)

measurement_time
2024-01-01           NaN
2024-01-02           NaN
2024-01-03           NaN
2024-01-04           NaN
2024-01-05    180.442000
2024-01-06    180.413750
2024-01-07    180.575250
2024-01-08    180.744833
2024-01-09    180.865000
2024-01-10    181.201083
Freq: D, Name: temperature, dtype: float64

In [82]:
daily_production

,생산건수,총생산량,총불량수,불량률
production_date,,,,
2024-01-01,20,2019,114,5.65
2024-01-02,22,2380,128,5.38
2024-01-03,16,1848,95,5.14
2024-01-04,22,2358,128,5.43
2024-01-05,20,2330,123,5.28
...,...,...,...,...
2024-03-27,22,2496,386,15.46
2024-03-28,20,2107,310,14.71
2024-03-29,22,2551,391,15.33


In [88]:
daily_production.shift(-1)

,생산건수,총생산량,총불량수,불량률
production_date,,,,
2024-01-01,22.0,2380.0,128.0,5.38
2024-01-02,16.0,1848.0,95.0,5.14
2024-01-03,22.0,2358.0,128.0,5.43
2024-01-04,20.0,2330.0,123.0,5.28
2024-01-05,22.0,2503.0,138.0,5.51
...,...,...,...,...
2024-03-27,20.0,2107.0,310.0,14.71
2024-03-28,22.0,2551.0,391.0,15.33
2024-03-29,20.0,2050.0,301.0,14.68


In [89]:
daily_production - daily_production.shift(1)

,생산건수,총생산량,총불량수,불량률
production_date,,,,
2024-01-01,NaN,NaN,NaN,NaN
2024-01-02,2.0,361.0,14.0,-0.27
2024-01-03,-6.0,-532.0,-33.0,-0.24
2024-01-04,6.0,510.0,33.0,0.29
2024-01-05,-2.0,-28.0,-5.0,-0.15
...,...,...,...,...
2024-03-27,4.0,364.0,82.0,1.20
2024-03-28,-2.0,-389.0,-76.0,-0.75
2024-03-29,2.0,444.0,81.0,0.62


In [90]:
# diff 함수

In [92]:
# 전일 대비 증감량
daily_production.diff(7)

,생산건수,총생산량,총불량수,불량률
production_date,,,,
2024-01-01,NaN,NaN,NaN,NaN
2024-01-02,NaN,NaN,NaN,NaN
2024-01-03,NaN,NaN,NaN,NaN
2024-01-04,NaN,NaN,NaN,NaN
2024-01-05,NaN,NaN,NaN,NaN
...,...,...,...,...
2024-03-27,2.0,376.0,70.0,0.55
2024-03-28,-2.0,-333.0,-42.0,0.28
2024-03-29,4.0,605.0,98.0,0.27


In [95]:
# 변화율 계산은 pct_change() 이용 
daily_production.pct_change() * 100

,생산건수,총생산량,총불량수,불량률
production_date,,,,
2024-01-01,NaN,NaN,NaN,NaN
2024-01-02,10.000000,17.880139,12.280702,-4.778761
2024-01-03,-27.272727,-22.352941,-25.781250,-4.460967
2024-01-04,37.500000,27.597403,34.736842,5.642023
2024-01-05,-9.090909,-1.187447,-3.906250,-2.762431
...,...,...,...,...
2024-03-27,22.222222,17.073171,26.973684,8.415147
2024-03-28,-9.090909,-15.584936,-19.689119,-4.851229
2024-03-29,10.000000,21.072615,26.129032,4.214820


In [96]:
# 이상치 탐지

In [ ]:
# IQR
# Z-score
# 3-Sigma

In [98]:
# OEE  ( Overall Equipment Effectiveness ) 

In [99]:
# 가동률 * 성능률 * 양품률

In [ ]:
# 가동률 = 실제 가동시간 / 계획 가동시간 

In [100]:
# 성능률 = (실제 생산량 * 이상 사이클타임) / 실제 가동시간

In [101]:
# 양품률 = 양품수량 / 총생산량

In [102]:
### OEE 구해서 50% 미만 => 현재의 설비 효율 개선하자.
### 85 이상 => 설비 증설 검토 

In [103]:
### 가동률이 낮다 => 고장 / 정지 시간을 줄이기 위한 방법을 찾아야한다.

In [104]:
### 성능률 낮다 => 속도 개선, 교체 시간 단축 위한 방법을 찾아야 한다.

In [105]:
### 양품률 낮다 => 품질 개선, 재작업 감소하는 방법을 찾아야 한다.

In [106]:
### 벤치마킹 가능하다 =>  설비간 비교를 통해서, 우수 설비를 찾아낸다.

In [107]:
### 공장간 비교가 가능하다. 우수공장은 Best Practice 가 된다.

In [ ]:
# OEE 계산결과 
# 85% 이상 : 월클 수준
# 70~84%  : 양호
# 60~69%  : 보통. 개선 과제 도출 
# 60% 미만 : 개선필요. 긴급조치

In [109]:
production_df['불량률'] = production_df['defect_quantity'] / production_df['actual_quantity'] * 100

In [111]:
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at', '생산효율',
       '효율 등급', '불량률'],
      dtype='object')

In [113]:
production_df[ [ 'target_quantity', 'actual_quantity', 'cycle_time','불량률']].corr()

,target_quantity,actual_quantity,cycle_time,불량률
target_quantity,1.000000,0.930913,0.016590,-0.075707
actual_quantity,0.930913,1.000000,-0.085386,-0.117936
cycle_time,0.016590,-0.085386,1.000000,0.058185
불량률,-0.075707,-0.117936,0.058185,1.000000
